In [1]:
# =========================================================
# UK HOUSING AFFORDABILITY - CLEAN DATA PIPELINE
# =========================================================

import os
import pandas as pd
from google.colab import drive

# =========================================================
# 1. MOUNT DRIVE
# =========================================================

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# =========================================================
# 2. DEFINE PROJECT PATHS
# =========================================================

project_path = "/content/drive/MyDrive/DataAnalytics/UKHousingAffordabilityAnalysis"

data_path = os.path.join(project_path, "Data")
raw_data_path = os.path.join(data_path, "Raw")
interim_data_path = os.path.join(data_path, "Interim")
processed_data_path = os.path.join(data_path, "Processed")

outputs_path = os.path.join(project_path, "Outputs")
metrics_path = os.path.join(outputs_path, "Metrics")
tables_path = os.path.join(outputs_path, "Tables")
figures_path = os.path.join(outputs_path, "Figures")
tableau_path = os.path.join(outputs_path, "Tableau")

for path in [
    raw_data_path,
    interim_data_path,
    processed_data_path,
    metrics_path,
    tables_path,
    figures_path,
    tableau_path,
]:
    os.makedirs(path, exist_ok=True)



In [3]:
# =========================================================
# 3. FILE PATHS
# =========================================================

earnings_workbook = os.path.join(
    raw_data_path,
    "aff2ratioofhousepricetoresidencebasedearnings.xlsx"
)

income_workbook = os.path.join(
    raw_data_path,
    "housingpurchaseaffordabilityratioslocalauthorityareasinenglandandwales.xlsx"
)

print("Earnings workbook exists:", os.path.exists(earnings_workbook))
print("Income workbook exists:", os.path.exists(income_workbook))



Earnings workbook exists: True
Income workbook exists: True


In [4]:
# =========================================================
# 4. HELPER FUNCTION - LOAD ONS SHEET
# =========================================================

def load_ons_sheet(file_path, sheet_name, header_keyword):
    preview = pd.read_excel(file_path, sheet_name=sheet_name, header=None)

    header_row = None
    for i in range(15):
        row_values = preview.iloc[i].astype(str).str.lower().tolist()
        if header_keyword.lower() in row_values:
            header_row = i
            break

    if header_row is None:
        raise ValueError(f"Header row not found in sheet: {sheet_name}")

    df = pd.read_excel(file_path, sheet_name=sheet_name, header=header_row)

    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace("\n", " ", regex=False)
    )

    df = df.dropna(how="all").reset_index(drop=True)
    return df



In [5]:
# =========================================================
# 5. LOAD SHEETS
# =========================================================

table_5a = load_ons_sheet(
    file_path=earnings_workbook,
    sheet_name="5a",
    header_keyword="Local authority code"
)

table_5b = load_ons_sheet(
    file_path=earnings_workbook,
    sheet_name="5b",
    header_keyword="Local authority code"
)

table_5c = load_ons_sheet(
    file_path=earnings_workbook,
    sheet_name="5c",
    header_keyword="Local authority code"
)

table_1 = load_ons_sheet(
    file_path=income_workbook,
    sheet_name="1",
    header_keyword="Region code"
)

print("Table 5a shape:", table_5a.shape)
print("Table 5b shape:", table_5b.shape)
print("Table 5c shape:", table_5c.shape)
print("Table 1 shape:", table_1.shape)

# =========================================================
# 6. REMOVE UNNECESSARY COLUMN
# =========================================================

table_5c = table_5c.drop(columns=["5-year average"], errors="ignore")



Table 5a shape: (318, 27)
Table 5b shape: (318, 27)
Table 5c shape: (318, 28)
Table 1 shape: (317, 30)


In [6]:
# =========================================================
# 7. RESHAPE DATASET 1: WIDE -> LONG
# =========================================================

id_vars_dataset1 = [
    "Country/Region code",
    "Country/Region name",
    "Local authority code",
    "Local authority name"
]

# Melt each earnings-based table from wide to long format
price_long = table_5a.melt(
    id_vars=id_vars_dataset1,
    var_name="Year",
    value_name="Median_House_Price"
)

earnings_long = table_5b.melt(
    id_vars=id_vars_dataset1,
    var_name="Year",
    value_name="Median_Earnings"
)

ratio_long = table_5c.melt(
    id_vars=id_vars_dataset1,
    var_name="Year",
    value_name="Affordability_Ratio_Earnings"
)

# Standardise year columns
# These tables use calendar years such as 2002, 2003, ..., 2024
for df_long in [price_long, earnings_long, ratio_long]:
    df_long["Year"] = (
        df_long["Year"]
        .astype(str)
        .str.extract(r"(\d{4})")[0]
        .astype(int)
    )

# Optional quick checks
print("Price long shape:", price_long.shape)
print("Earnings long shape:", earnings_long.shape)
print("Ratio long shape:", ratio_long.shape)

print("\nYear range checks:")
print("Price:", price_long["Year"].min(), "to", price_long["Year"].max())
print("Earnings:", earnings_long["Year"].min(), "to", earnings_long["Year"].max())
print("Ratio:", ratio_long["Year"].min(), "to", ratio_long["Year"].max())

Price long shape: (7314, 6)
Earnings long shape: (7314, 6)
Ratio long shape: (7314, 6)

Year range checks:
Price: 2002 to 2024
Earnings: 2002 to 2024
Ratio: 2002 to 2024


In [7]:
# =========================================================
# 8. MERGE DATASET 1 TABLES
# =========================================================

dataset1 = (
    price_long
    .merge(
        earnings_long,
        on=id_vars_dataset1 + ["Year"],
        how="inner",
        validate="one_to_one"
    )
    .merge(
        ratio_long,
        on=id_vars_dataset1 + ["Year"],
        how="inner",
        validate="one_to_one"
    )
    .sort_values(["Local authority code", "Year"])
    .reset_index(drop=True)
)

print("Dataset1 shape:", dataset1.shape)
print("Unique local authorities:", dataset1["Local authority code"].nunique())
print("Year range:", dataset1["Year"].min(), "to", dataset1["Year"].max())
print("Duplicate rows:", dataset1.duplicated(subset=["Local authority code", "Year"]).sum())

Dataset1 shape: (7314, 8)
Unique local authorities: 318
Year range: 2002 to 2024
Duplicate rows: 0


In [8]:
# =========================================================
# 9. RESHAPE DATASET 2: WIDE -> LONG
# =========================================================

income_long = table_1.melt(
    id_vars=[
        "Region code",
        "Region name",
        "Local authority code",
        "Local authority name"
    ],
    var_name="Year",
    value_name="Affordability_Ratio_Income"
)

# Robust financial year conversion
income_long["Year"] = (
    income_long["Year"]
    .astype(str)
    .str.extract(r"/(\d{2})$")[0]
    .astype(int)
)

income_long["Year"] = income_long["Year"].apply(
    lambda x: 2000 + x if x < 90 else 1900 + x
)


In [9]:

# =========================================================
# 10. MERGE BOTH DATASETS
# =========================================================

final_df = dataset1.merge(
    income_long[
        ["Local authority code", "Year", "Region code", "Region name", "Affordability_Ratio_Income"]
    ],
    on=["Local authority code", "Year"],
    how="left"
)

print("Merged dataset shape:", final_df.shape)
print("Missing income values:", final_df["Affordability_Ratio_Income"].isna().sum())

Merged dataset shape: (7314, 11)
Missing income values: 23


In [10]:
missing_income_rows = final_df[final_df["Affordability_Ratio_Income"].isna()][
    ["Local authority code", "Local authority name"]
].drop_duplicates()

print(missing_income_rows)

     Local authority code Local authority name
1127            E06000053      Isles of Scilly


In [11]:
dataset1_las = dataset1[["Local authority code", "Local authority name"]].drop_duplicates()
income_las = income_long[["Local authority code", "Local authority name"]].drop_duplicates()

missing_in_income = dataset1_las.merge(
    income_las,
    on=["Local authority code", "Local authority name"],
    how="left",
    indicator=True
)

missing_in_income = missing_in_income[missing_in_income["_merge"] == "left_only"]

print("Authorities in dataset1 but not in income dataset:")
print(missing_in_income)

Authorities in dataset1 but not in income dataset:
   Local authority code Local authority name     _merge
49            E06000053      Isles of Scilly  left_only


In [12]:
# =========================================================
# 11. DATA TYPE CLEANING
# =========================================================

numeric_cols = [
    "Median_House_Price",
    "Median_Earnings",
    "Affordability_Ratio_Earnings",
    "Affordability_Ratio_Income"
]

# Convert numeric columns safely
for col in numeric_cols:
    final_df[col] = (
        final_df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("..", "", regex=False)
    )
    final_df[col] = pd.to_numeric(final_df[col], errors="coerce")

# Convert Year safely
final_df["Year"] = pd.to_numeric(final_df["Year"], errors="coerce")

# Optional: ensure integer type if no missing values
if final_df["Year"].isna().sum() == 0:
    final_df["Year"] = final_df["Year"].astype(int)

# =========================================================
# VALIDATION CHECKS
# =========================================================

print("\nData types:")
print(final_df.dtypes)

print("\nMissing values after conversion:")
print(final_df[numeric_cols + ["Year"]].isna().sum())


Data types:
Country/Region code              object
Country/Region name              object
Local authority code             object
Local authority name             object
Year                              int64
Median_House_Price                int64
Median_Earnings                 float64
Affordability_Ratio_Earnings    float64
Region code                      object
Region name                      object
Affordability_Ratio_Income      float64
dtype: object

Missing values after conversion:
Median_House_Price               0
Median_Earnings                 59
Affordability_Ratio_Earnings    59
Affordability_Ratio_Income      71
Year                             0
dtype: int64


In [13]:
# =========================================================
# 12. FILL MISSING REGION VALUES FROM DATASET 1
# =========================================================

final_df["Region code"] = final_df["Region code"].fillna(final_df["Country/Region code"])
final_df["Region name"] = final_df["Region name"].fillna(final_df["Country/Region name"])

print("Missing Region code:", final_df["Region code"].isna().sum())
print("Missing Region name:", final_df["Region name"].isna().sum())

Missing Region code: 0
Missing Region name: 0


In [14]:
# =========================================================
# 13. CREATE KPI + SAVE KPI OUTPUT
# =========================================================

# Create affordability gap KPI
final_df["Affordability_Gap"] = (
    final_df["Affordability_Ratio_Income"] -
    final_df["Affordability_Ratio_Earnings"]
)

# Quick KPI audit
print("Missing Affordability_Gap:", final_df["Affordability_Gap"].isna().sum())

print("\nAffordability_Gap summary:")
print(final_df["Affordability_Gap"].describe())

# Save KPI-focused table
kpi_output = final_df[
    [
        "Local authority code",
        "Local authority name",
        "Region code",
        "Region name",
        "Year",
        "Affordability_Ratio_Earnings",
        "Affordability_Ratio_Income",
        "Affordability_Gap"
    ]
].copy()

kpi_output.to_csv(
    os.path.join(tables_path, "Affordability_Gap_KPI_Table.csv"),
    index=False
)

print("\nKPI table saved:")
print(os.path.join(tables_path, "Affordability_Gap_KPI_Table.csv"))

Missing Affordability_Gap: 88

Affordability_Gap summary:
count    7226.000000
mean        0.749903
std         1.557011
min        -4.100000
25%        -0.070000
50%         0.480000
75%         1.190000
max        21.320000
Name: Affordability_Gap, dtype: float64

KPI table saved:
/content/drive/MyDrive/DataAnalytics/UKHousingAffordabilityAnalysis/Outputs/Tables/Affordability_Gap_KPI_Table.csv


In [15]:
# =========================================================
# 14. QUALITY CHECKS BEFORE DROPPING
# =========================================================

final_df_raw = final_df.copy()
initial_rows = len(final_df)

rows_missing_earnings = final_df["Median_Earnings"].isna().sum()
rows_missing_ratio_earnings = final_df["Affordability_Ratio_Earnings"].isna().sum()
rows_missing_ratio_income = final_df["Affordability_Ratio_Income"].isna().sum()

rows_missing_core = final_df[
    final_df["Median_Earnings"].isna() |
    final_df["Affordability_Ratio_Earnings"].isna()
].shape[0]



In [16]:
# =========================================================
# 14. QUALITY CHECKS BEFORE DROPPING
# =========================================================

final_df_raw = final_df.copy()
initial_rows = len(final_df)

rows_missing_earnings = final_df["Median_Earnings"].isna().sum()
rows_missing_ratio_earnings = final_df["Affordability_Ratio_Earnings"].isna().sum()
rows_missing_ratio_income = final_df["Affordability_Ratio_Income"].isna().sum()
rows_missing_gap = final_df["Affordability_Gap"].isna().sum()

rows_missing_core = final_df[
    final_df["Median_Earnings"].isna() |
    final_df["Affordability_Ratio_Earnings"].isna()
].shape[0]

print("\nMissing values audit before dropping:")
print("Rows missing Median_Earnings:", rows_missing_earnings)
print("Rows missing Affordability_Ratio_Earnings:", rows_missing_ratio_earnings)
print("Rows missing Affordability_Ratio_Income:", rows_missing_ratio_income)
print("Rows missing Affordability_Gap:", rows_missing_gap)
print("Rows missing core fields:", rows_missing_core)



Missing values audit before dropping:
Rows missing Median_Earnings: 59
Rows missing Affordability_Ratio_Earnings: 59
Rows missing Affordability_Ratio_Income: 71
Rows missing Affordability_Gap: 88
Rows missing core fields: 59


In [17]:
# =========================================================
# 15. DROP ROWS MISSING CORE FIELDS
# =========================================================

final_df = final_df.dropna(
    subset=["Median_Earnings", "Affordability_Ratio_Earnings"]
).copy()


In [18]:
# =========================================================
# 16. DROP REDUNDANT REGION COLUMNS
# =========================================================

final_df = final_df.drop(
    columns=["Country/Region code", "Country/Region name"],
    errors="ignore"
).reset_index(drop=True)

In [19]:
# =========================================================
# 17. FINAL AUDIT SUMMARY
# =========================================================

final_rows = len(final_df)
rows_removed = initial_rows - final_rows

missing_audit = pd.DataFrame({
    "Metric": [
        "Initial rows",
        "Rows missing Median_Earnings",
        "Rows missing Affordability_Ratio_Earnings",
        "Rows missing Affordability_Ratio_Income",
        "Rows missing Affordability_Gap",
        "Rows missing core fields dropped",
        "Rows removed total",
        "Final rows"
    ],
    "Value": [
        initial_rows,
        rows_missing_earnings,
        rows_missing_ratio_earnings,
        rows_missing_ratio_income,
        rows_missing_gap,
        rows_missing_core,
        rows_removed,
        final_rows
    ]
})

print("\nMissing Values Audit Summary:")
print(missing_audit)

print("\nFinal missing values:")
print(final_df.isna().sum())

print("\nDuplicate rows:", final_df.duplicated().sum())
print("Year range:", final_df["Year"].min(), "to", final_df["Year"].max())
print("Unique Local Authorities:", final_df["Local authority name"].nunique())
print("Unique Regions:", final_df["Region name"].nunique())

print("\nFinal dataset shape:", final_df.shape)
print("\nFinal dataset preview:")
print(final_df.head())

print("\nSummary statistics:")
print(final_df.describe())



Missing Values Audit Summary:
                                      Metric  Value
0                               Initial rows   7314
1               Rows missing Median_Earnings     59
2  Rows missing Affordability_Ratio_Earnings     59
3    Rows missing Affordability_Ratio_Income     71
4             Rows missing Affordability_Gap     88
5           Rows missing core fields dropped     59
6                         Rows removed total     59
7                                 Final rows   7255

Final missing values:
Local authority code             0
Local authority name             0
Year                             0
Median_House_Price               0
Median_Earnings                  0
Affordability_Ratio_Earnings     0
Region code                      0
Region name                      0
Affordability_Ratio_Income      29
Affordability_Gap               29
dtype: int64

Duplicate rows: 0
Year range: 2002 to 2024
Unique Local Authorities: 318
Unique Regions: 10

Final dataset shape: 

In [20]:
# =========================================================
# 18. SAVE OUTPUTS
# =========================================================

final_df.to_csv(
    os.path.join(processed_data_path, "UKHousingAffordabilityDataset.csv"),
    index=False
)

missing_audit.to_csv(
    os.path.join(tables_path, "MissingValuesAuditSummary.csv"),
    index=False
)

print("\nSaved files:")
print(os.path.join(processed_data_path, "UKHousingAffordabilityDataset.csv"))
print(os.path.join(tables_path, "MissingValuesAuditSummary.csv"))


Saved files:
/content/drive/MyDrive/DataAnalytics/UKHousingAffordabilityAnalysis/Data/Processed/UKHousingAffordabilityDataset.csv
/content/drive/MyDrive/DataAnalytics/UKHousingAffordabilityAnalysis/Outputs/Tables/MissingValuesAuditSummary.csv
